In [41]:
import pandas as pd
import numpy as np

Movies = pd.read_csv('../movies_Data/movies.csv')
Ratings = pd.read_csv('../movies_Data/ratings.csv')


In [42]:
Movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [43]:
Ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [23]:
Movies.isna().sum()
Ratings.isna().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [44]:
# 1. Compter les utilisateurs distincts par film
popularity = (
    Ratings.groupby("movieId")["userId"]
    .nunique()
    .reset_index(name="rating_count")
)

# 2. Ajouter les informations du film
ranking = popularity.merge(
    Movies,
    on="movieId",
    how="left",
    validate="one_to_one"
)

# 3. Classer par popularité décroissante
ranking = ranking.sort_values(
    by=["rating_count", "movieId"],
    ascending=[False, True]
)

# 4. Afficher les 10 premiers
ranking[["movieId", "title", "rating_count"]].head(10)

,movieId,title,rating_count
314,356,Forrest Gump (1994),329
277,318,"Shawshank Redemption, The (1994)",317
257,296,Pulp Fiction (1994),307
510,593,"Silence of the Lambs, The (1991)",279
1938,2571,"Matrix, The (1999)",278
224,260,Star Wars: Episode IV - A New Hope (1977),251
418,480,Jurassic Park (1993),238
97,110,Braveheart (1995),237
507,589,Terminator 2: Judgment Day (1991),224
461,527,Schindler's List (1993),220


In [46]:
#transformation de notre colone timestamp en datetime pour integrer la tendance temporelle non pas seulement popularite 
Ratings["datetime"] = pd.to_datetime(
    Ratings["timestamp"],
    unit="s",
    utc=True
)
Ratings=Ratings.sort_values(by='datetime', ascending=True)

In [47]:
import numpy as np

reference_date = Ratings["datetime"].max()
half_life_days = 30

# Conserver uniquement les évaluations connues à cette date
weighted_ratings = Ratings.loc[
    Ratings["datetime"] <= reference_date
].copy()

# Âge de chaque évaluation en jours
age_days = (
    reference_date - weighted_ratings["datetime"]
).dt.total_seconds() / 86400

# Une évaluation perd la moitié de son poids tous les 30 jours
weighted_ratings["time_weight"] = np.exp2(
    -age_days / half_life_days
)

# Somme des contributions par film
decayed_popularity = (
    weighted_ratings.groupby("movieId")["time_weight"]
    .sum()
    .reset_index(name="popularity_score")
)

decayed_ranking = (
    decayed_popularity
    .merge(Movies, on="movieId", validate="one_to_one")
    .sort_values(
        ["popularity_score", "movieId"],
        ascending=[False, True]
    )
)

decayed_ranking[
    ["movieId", "title", "popularity_score"]
].head(10)

,movieId,title,popularity_score
9691,187593,Deadpool 2 (2018),4.650477
7355,79132,Inception (2010),4.599866
8676,122916,Thor: Ragnarok (2017),4.462210
6693,58559,"Dark Knight, The (2008)",4.394512
8674,122906,Black Panther (2017),4.104977
8675,122912,Avengers: Infinity War - Part I (2018),4.092880
4791,7153,"Lord of the Rings: The Return of the King, The...",4.079527
4131,5952,"Lord of the Rings: The Two Towers, The (2002)",3.927997
3633,4993,"Lord of the Rings: The Fellowship of the Ring,...",3.910318
1938,2571,"Matrix, The (1999)",3.847310


## RAnking base sur la popularite seule ou meme avec tendance temporelle n'a aucun edge nos model ne peuvent etre evalue 
donc les classement faite base sur ces deux variable seules ne sont pas fialbe pour cela 
learning to rank we integrate the Machine learning to predict the score that we will use to rank 
we use time series split to ensure no data leakage 

In [ ]:
# ============================================================
# 1. IMPORTS ET PARAMÈTRES — TOUT EST DÉCLARÉ ICI
# ============================================================

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import ndcg_score


MOVIES_PATH = "../movies_Data/movies.csv"
RATINGS_PATH = "../movies_Data/ratings.csv"

HORIZON_DAYS = 30
STEP_DAYS = 30
WARMUP_DAYS = 180

MIN_PAST_USERS = 5
POSITIVE_THRESHOLD = 4.0
HALF_LIFE_DAYS = 30

TOP_K = 10
RANDOM_STATE = 42

HORIZON = pd.Timedelta(days=HORIZON_DAYS)

# Niveaux de pertinence fixés avant l'entraînement :
# 0 positif futur    -> niveau 0
# 1 à 2              -> niveau 1
# 3 à 6              -> niveau 2
# 7 à 14             -> niveau 3
# 15 ou plus         -> niveau 4
RELEVANCE_BINS = np.array([1, 3, 7, 15])
LABEL_GAINS = np.array([0, 1, 3, 7, 15])

FEATURE_COLUMNS = [
    "past_users",
    "past_mean_rating",
    "past_positive_users",
    "users_7d",
    "users_30d",
    "users_90d",
    "mean_rating_30d",
    "positive_users_30d",
    "trend_30d",
    "decayed_popularity",
    "days_since_last_rating",
]


# ============================================================
# 2. IMPORTATION ET CONTRÔLES
# ============================================================

movies = pd.read_csv(MOVIES_PATH)
ratings = pd.read_csv(RATINGS_PATH)

required_movies = {"movieId", "title"}
required_ratings = {"userId", "movieId", "rating", "timestamp"}

if not required_movies.issubset(movies.columns):
    raise ValueError(
        f"Colonnes requises dans movies : {required_movies}"
    )

if not required_ratings.issubset(ratings.columns):
    raise ValueError(
        f"Colonnes requises dans ratings : {required_ratings}"
    )

# On repart des CSV originaux : timestamp Unix en secondes.
ratings["timestamp"] = pd.to_numeric(
    ratings["timestamp"], errors="raise"
)

ratings["rating"] = pd.to_numeric(
    ratings["rating"], errors="raise"
)

ratings["datetime"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s",
    utc=True,
    errors="raise"
)

if movies[["movieId", "title"]].isna().any().any():
    raise ValueError("Valeurs manquantes dans movies.")

if ratings[
    ["userId", "movieId", "rating", "datetime"]
].isna().any().any():
    raise ValueError("Valeurs manquantes dans ratings.")

if not movies["movieId"].is_unique:
    raise ValueError("movieId doit être unique dans movies.")

if not ratings["movieId"].isin(movies["movieId"]).all():
    raise ValueError("Certains films de ratings sont absents de movies.")

# Hypothèse de cette implémentation :
# une seule évaluation par utilisateur et par film.
if ratings.duplicated(["userId", "movieId"]).any():
    raise ValueError(
        "Plusieurs évaluations existent pour un même couple "
        "utilisateur-film. Définir leur traitement avant de continuer."
    )

# Hypothèse : dataset noté sur 5.
if not ratings["rating"].between(0, 5).all():
    raise ValueError(
        "Les notes ne sont pas sur une échelle comprise entre 0 et 5."
    )

ratings = ratings.sort_values(
    "datetime"
).reset_index(drop=True)

print("Nombre de films :", len(movies))
print("Nombre d'évaluations :", len(ratings))
print("Première évaluation :", ratings["datetime"].min())
print("Dernière évaluation :", ratings["datetime"].max())


# ============================================================
# 3. FEATURES : UNIQUEMENT LES INFORMATIONS ANTÉRIEURES À t
# ============================================================

def build_features(
    ratings_data,
    reference_date,
    min_past_users,
    positive_threshold,
    half_life_days
):
    past = ratings_data.loc[
        ratings_data["datetime"] < reference_date
    ].copy()

    features = past.groupby("movieId").agg(
        past_users=("userId", "nunique"),
        past_mean_rating=("rating", "mean"),
        last_rating_date=("datetime", "max")
    )

    # Éligibilité fondée exclusivement sur le passé.
    features = features.loc[
        features["past_users"] >= min_past_users
    ].copy()

    positive_past = past.loc[
        past["rating"] >= positive_threshold
    ]

    features["past_positive_users"] = (
        positive_past.groupby("movieId")["userId"].nunique()
    )

    for days in [7, 30, 90]:
        window_start = reference_date - pd.Timedelta(days=days)

        window = past.loc[
            past["datetime"] >= window_start
        ]

        features[f"users_{days}d"] = (
            window.groupby("movieId")["userId"].nunique()
        )

        if days == 30:
            features["mean_rating_30d"] = (
                window.groupby("movieId")["rating"].mean()
            )

            positive_window = window.loc[
                window["rating"] >= positive_threshold
            ]

            features["positive_users_30d"] = (
                positive_window.groupby("movieId")["userId"]
                .nunique()
            )

    # Fenêtre précédente : [t - 60 jours, t - 30 jours[
    previous_window = past.loc[
        (
            past["datetime"]
            >= reference_date - pd.Timedelta(days=60)
        )
        & (
            past["datetime"]
            < reference_date - pd.Timedelta(days=30)
        )
    ]

    previous_users = (
        previous_window.groupby("movieId")["userId"]
        .nunique()
        .reindex(features.index, fill_value=0)
    )

    features["trend_30d"] = (
        features["users_30d"].fillna(0) - previous_users
    )

    age_days = (
        reference_date - past["datetime"]
    ).dt.total_seconds() / 86400.0

    past["time_weight"] = np.exp2(
        -age_days / half_life_days
    )

    features["decayed_popularity"] = (
        past.groupby("movieId")["time_weight"].sum()
    )

    features["days_since_last_rating"] = (
        reference_date - features["last_rating_date"]
    ).dt.total_seconds() / 86400.0

    # Si aucun avis récent : moyenne historique du film.
    features["mean_rating_30d"] = (
        features["mean_rating_30d"]
        .fillna(features["past_mean_rating"])
    )

    features = (
        features.drop(columns="last_rating_date")
        .fillna(0)
        .reset_index()
    )

    features["reference_date"] = reference_date

    return features


# ============================================================
# 4. CIBLE : INFORMATIONS FUTURES, JAMAIS UTILISÉES COMME FEATURES
# ============================================================

def build_snapshot(
    ratings_data,
    reference_date,
    horizon,
    min_past_users,
    positive_threshold,
    half_life_days,
    relevance_bins
):
    snapshot = build_features(
        ratings_data=ratings_data,
        reference_date=reference_date,
        min_past_users=min_past_users,
        positive_threshold=positive_threshold,
        half_life_days=half_life_days
    )

    # Fenêtre cible : [t, t + horizon[
    future_positive = ratings_data.loc[
        (ratings_data["datetime"] >= reference_date)
        & (ratings_data["datetime"] < reference_date + horizon)
        & (ratings_data["rating"] >= positive_threshold)
    ]

    future_counts = (
        future_positive.groupby("movieId")["userId"].nunique()
    )

    snapshot["future_positive_users"] = (
        snapshot["movieId"]
        .map(future_counts)
        .fillna(0)
        .astype(int)
    )

    snapshot["relevance"] = np.digitize(
        snapshot["future_positive_users"].to_numpy(),
        bins=relevance_bins
    ).astype(int)

    snapshot["label_end"] = reference_date + horizon

    return snapshot


# ============================================================
# 5. CRÉATION DU DATASET : UNE LIGNE = UN FILM À UNE DATE
# ============================================================

first_reference = (
    ratings["datetime"].min()
    + pd.Timedelta(days=WARMUP_DAYS)
)

last_reference = ratings["datetime"].max() - HORIZON

reference_dates = pd.date_range(
    start=first_reference,
    end=last_reference,
    freq=pd.Timedelta(days=STEP_DAYS)
)

snapshots = []

for reference_date in reference_dates:
    snapshot = build_snapshot(
        ratings_data=ratings,
        reference_date=reference_date,
        horizon=HORIZON,
        min_past_users=MIN_PAST_USERS,
        positive_threshold=POSITIVE_THRESHOLD,
        half_life_days=HALF_LIFE_DAYS,
        relevance_bins=RELEVANCE_BINS
    )

    if len(snapshot) >= TOP_K:
        snapshots.append(snapshot)

if not snapshots:
    raise ValueError(
        "Aucune observation exploitable : vérifier la durée "
        "du dataset et le nombre de films éligibles."
    )

dataset = (
    pd.concat(snapshots, ignore_index=True)
    .sort_values(["reference_date", "movieId"])
    .reset_index(drop=True)
)

print("\nNombre d'observations :", len(dataset))
print(
    "Nombre de dates de classement :",
    dataset["reference_date"].nunique()
)


# ============================================================
# 6. SPLIT TEMPOREL + EXCLUSION DES CIBLES QUI DÉBORDENT
# ============================================================

dates = pd.DatetimeIndex(
    dataset["reference_date"].unique()
).sort_values()

if len(dates) < 15:
    raise ValueError(
        "Moins de 15 dates de classement : "
        "historique insuffisant pour ce protocole."
    )

validation_start = dates[int(len(dates) * 0.60)]
test_start = dates[int(len(dates) * 0.80)]

train = dataset.loc[
    (dataset["reference_date"] < validation_start)
    & (dataset["label_end"] <= validation_start)
].copy()

validation = dataset.loc[
    (dataset["reference_date"] >= validation_start)
    & (dataset["reference_date"] < test_start)
    & (dataset["label_end"] <= test_start)
].copy()

test = dataset.loc[
    dataset["reference_date"] >= test_start
].copy()

for name, partition in [
    ("TRAIN", train),
    ("VALIDATION", validation),
    ("TEST", test)
]:
    if partition.empty:
        raise ValueError(f"La partition {name} est vide.")

    print(
        f"\n{name}",
        "\n  Première date :", partition["reference_date"].min(),
        "\n  Dernière date :", partition["reference_date"].max(),
        "\n  Groupes       :", partition["reference_date"].nunique(),
        "\n  Lignes        :", len(partition)
    )

assert (
    train["label_end"].max()
    <= validation["reference_date"].min()
)

assert (
    validation["label_end"].max()
    <= test["reference_date"].min()
)


# ============================================================
# 7. PRÉPARATION DES MATRICES ET DES GROUPES
# ============================================================

def prepare_partition(partition, feature_columns):
    ordered = partition.sort_values(
        ["reference_date", "movieId"]
    ).reset_index(drop=True)

    X = ordered[feature_columns].astype(float)
    y = ordered["relevance"].astype(int)

    groups = (
        ordered.groupby("reference_date", sort=False)
        .size()
        .to_numpy()
    )

    if not np.isfinite(X.to_numpy()).all():
        raise ValueError("Features contenant NaN ou infini.")

    assert groups.sum() == len(ordered)

    return ordered, X, y, groups


train, X_train, y_train, train_groups = prepare_partition(
    train, FEATURE_COLUMNS
)

validation, X_val, y_val, val_groups = prepare_partition(
    validation, FEATURE_COLUMNS
)

test, X_test, y_test, test_groups = prepare_partition(
    test, FEATURE_COLUMNS
)

forbidden_features = {
    "future_positive_users",
    "relevance",
    "label_end",
    "rating",
    "timestamp",
    "datetime",
    "reference_date"
}

assert not forbidden_features.intersection(X_train.columns)

for name, partition in [
    ("TRAIN", train),
    ("VALIDATION", validation)
]:
    informative_groups = (
        partition.groupby("reference_date")["relevance"]
        .nunique()
        .gt(1)
        .sum()
    )

    if informative_groups == 0:
        raise ValueError(
            f"{name} ne contient aucun groupe avec des pertinences "
            "différentes. Vérifier les cibles et leurs seuils."
        )

print("\nFeatures utilisées :", X_train.columns.tolist())
print("\nRépartition des labels TRAIN :")
print(y_train.value_counts().sort_index())


# ============================================================
# 8. ENTRAÎNEMENT — EARLY STOPPING SUR VALIDATION UNIQUEMENT
# ============================================================

model = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    label_gain=LABEL_GAINS.tolist(),
    lambdarank_truncation_level=TOP_K + 3,
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=15,
    max_depth=5,
    min_child_samples=30,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

model.fit(
    X_train,
    y_train,
    group=train_groups,
    eval_set=[(X_val, y_val)],
    eval_group=[val_groups],
    eval_at=[TOP_K],
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True
        ),
        lgb.log_evaluation(period=50)
    ]
)

print("\nMeilleure itération :", model.best_iteration_)


# ============================================================
# 9. PRÉDICTION TEST — UNIQUEMENT À PARTIR DE X_test
# ============================================================

test["predicted_score"] = model.predict(
    X_test,
    num_iteration=model.best_iteration_
)


# ============================================================
# 10. ÉVALUATION PAR DATE ET COMPARAISON AUX BASELINES
# ============================================================

def evaluate_ranking(
    predictions,
    score_column,
    label_gains,
    top_k
):
    results = []

    for reference_date, group in predictions.groupby("reference_date"):
        future_counts = group["future_positive_users"].to_numpy()

        # Aucune pertinence observable : métrique non informative.
        if future_counts.sum() == 0:
            continue

        gains = label_gains[
            group["relevance"].to_numpy(dtype=int)
        ]

        scores = group[score_column].to_numpy(dtype=float)

        ndcg = ndcg_score(
            gains.reshape(1, -1),
            scores.reshape(1, -1),
            k=top_k
        )

        top = (
            group.sort_values(
                [score_column, "movieId"],
                ascending=[False, True]
            )
            .head(top_k)
        )

        capture = (
            top["future_positive_users"].sum()
            / future_counts.sum()
        )

        results.append({
            "reference_date": reference_date,
            "ndcg": ndcg,
            "positive_capture": capture
        })

    return pd.DataFrame(
        results,
        columns=["reference_date", "ndcg", "positive_capture"]
    )


methods = {
    "LambdaRank": "predicted_score",
    "Popularité cumulée": "past_users",
    "Popularité récente 30j": "users_30d",
    "Popularité décroissante": "decayed_popularity"
}

evaluation_details = {}
summary_rows = []

for method_name, score_column in methods.items():
    metrics = evaluate_ranking(
        predictions=test,
        score_column=score_column,
        label_gains=LABEL_GAINS,
        top_k=TOP_K
    )

    evaluation_details[method_name] = metrics

    summary_rows.append({
        "method": method_name,
        f"NDCG@{TOP_K}": metrics["ndcg"].mean(),
        "NDCG_std": metrics["ndcg"].std(),
        f"positive_capture@{TOP_K}": (
            metrics["positive_capture"].mean()
        ),
        "evaluated_dates": len(metrics)
    })

comparison = (
    pd.DataFrame(summary_rows)
    .sort_values(f"NDCG@{TOP_K}", ascending=False)
    .reset_index(drop=True)
)

if comparison["evaluated_dates"].max() == 0:
    raise ValueError(
        "Aucune cible positive dans le test : "
        "évaluation impossible avec ces paramètres."
    )

print("\nCOMPARAISON SUR LE TEST")
print(comparison.round(4).to_string(index=False))

empty_target_dates = (
    test.groupby("reference_date")["future_positive_users"]
    .sum()
    .eq(0)
    .sum()
)

print(
    "\nDates exclues de l'évaluation "
    "(aucun évaluateur positif futur) :",
    empty_target_dates
)


# ============================================================
# 11. AFFICHER LE TOP K D'UNE DATE DU TEST
# ============================================================

display_date = test["reference_date"].max()

top_movies = (
    test.loc[test["reference_date"] == display_date]
    .sort_values(
        ["predicted_score", "movieId"],
        ascending=[False, True]
    )
    .head(TOP_K)
    .merge(
        movies[["movieId", "title"]],
        on="movieId",
        how="left",
        validate="one_to_one"
    )
)

print("\nClassement prédit à la date :", display_date)

print(
    top_movies[
        [
            "title",
            "predicted_score",
            "future_positive_users",
            "relevance"
        ]
    ].to_string(index=False)
)


Nombre de films : 9742
Nombre d'évaluations : 100836
Première évaluation : 1996-03-29 18:36:55+00:00
Dernière évaluation : 2018-09-24 14:27:30+00:00

Nombre d'observations : 549720
Nombre de dates de classement : 267

TRAIN 
  Première date : 1996-09-25 18:36:55+00:00 
  Dernière date : 2009-10-17 18:36:55+00:00 
  Groupes       : 160 
  Lignes        : 230186

VALIDATION 
  Première date : 2009-11-16 18:36:55+00:00 
  Dernière date : 2014-02-23 18:36:55+00:00 
  Groupes       : 53 
  Lignes        : 146671

TEST 
  Première date : 2014-03-25 18:36:55+00:00 
  Dernière date : 2018-08-01 18:36:55+00:00 
  Groupes       : 54 
  Lignes        : 172863

Features utilisées : ['past_users', 'past_mean_rating', 'past_positive_users', 'users_7d', 'users_30d', 'users_90d', 'mean_rating_30d', 'positive_users_30d', 'trend_30d', 'decayed_popularity', 'days_since_last_rating']

Répartition des labels TRAIN :
relevance
0    214676
1     14893
2       604
3        13
Name: count, dtype: int64


c:\Users\user\Desktop\projects\Systeme de recommendation\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[50]	valid_0's ndcg@10: 0.339558
[100]	valid_0's ndcg@10: 0.383218
[150]	valid_0's ndcg@10: 0.376676
Early stopping, best iteration is:
[131]	valid_0's ndcg@10: 0.384869
Evaluated only: ndcg@10

Meilleure itération : 131

COMPARAISON SUR LE TEST
                 method  NDCG@10  NDCG_std  positive_capture@10  evaluated_dates
             LambdaRank   0.4600    0.2549               0.0545               54
     Popularité cumulée   0.4321    0.2357               0.0516               54
Popularité décroissante   0.4096    0.2412               0.0483               54
 Popularité récente 30j   0.2994    0.2023               0.0351               54

Dates exclues de l'évaluation (aucun évaluateur positif futur) : 0

Classement prédit à la date : 2018-08-01 18:36:55+00:00
                                                                         title  predicted_score  future_positive_users  relevance
                                 

In [51]:
from pathlib import Path
import joblib

artifact_directory = Path("artifacts")
artifact_directory.mkdir(parents=True, exist_ok=True)

artifact = {
    "model": model,
    "feature_columns": FEATURE_COLUMNS,
    "min_past_users": MIN_PAST_USERS,
    "positive_threshold": POSITIVE_THRESHOLD,
    "half_life_days": HALF_LIFE_DAYS,
    "horizon_days": HORIZON_DAYS,
    "best_iteration": model.best_iteration_,
}

artifact_path = artifact_directory / "movie_ranker.joblib"

joblib.dump(artifact, artifact_path)

print("Modèle sauvegardé :", artifact_path.resolve())

Modèle sauvegardé : C:\Users\user\Desktop\projects\Systeme de recommendation\Movies_recommendation\artifacts\movie_ranker.joblib
